# CUDA Graphs — Eliminating CPU Launch Overhead

**Module 21** | Prerequisites: Modules 07, 08, 12 | Time: ~2 hours

This notebook explores CUDA Graphs — a technique for capturing sequences of GPU operations and replaying them with minimal CPU overhead, often yielding 2–10x throughput improvements for inference.

**What you'll learn:**
- What CUDA Graphs are and why they eliminate CPU bottlenecks
- The basic capture/replay API
- Static inputs requirement and the warmup pattern
- Benchmarking eager vs graph replay
- `torch.compile` with `reduce-overhead` mode
- `make_graphed_callables` for quick experiments
- Limitations and when to use (or not use) CUDA Graphs

In [ ]:
import torch
import torch.nn as nn
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

HAS_CUDA = torch.cuda.is_available()

## What Are CUDA Graphs?

Every GPU operation (kernel launch, memory copy) requires the CPU to submit work to the CUDA driver. For small/medium kernels, this **CPU launch overhead** (5–15 µs per kernel) can dominate total runtime.

CUDA Graphs solve this by **recording** a sequence of GPU operations during a capture phase, then **replaying** the entire sequence with a single CPU launch.

```
Normal execution:               CUDA Graph replay:

CPU: launch K1                  CPU: launch graph ──────────────┐
     wait...                                                      │
     launch K2                  GPU: K1 → K2 → K3 → K4 → K5     │
     wait...                         (all pre-recorded)           │
     launch K3                                                    │
     wait...                    1 CPU launch instead of 5         │
     launch K4                  ──────────────────────────────┘
     wait...
     launch K5

GPU: [K1] gap [K2] gap [K3]    GPU: [K1][K2][K3][K4][K5]
     (idle gaps)                     (back-to-back, no gaps)
```

**Key insight:** The graph captures **memory addresses**, not tensor values. This is why inputs must be pre-allocated “static” tensors.

## Why CUDA Graphs Matter — The Numbers

```
Without CUDA Graphs (100 small kernels):
  CPU overhead:  100 x 10 us = 1,000 us
  GPU compute:   100 x  3 us =   300 us
  Total:                        1,300 us   (23% GPU utilization)

With CUDA Graphs:
  CPU overhead:    1 x 10 us =    10 us
  GPU compute:   100 x  3 us =   300 us
  Total:                          310 us   (97% GPU utilization)
  Speedup:                        4.2x
```

| Scenario | Typical Speedup |
|----------|----------------|
| Small model inference (ResNet-18, batch=1) | 3–10x |
| Medium model inference (BERT, batch=8) | 2–5x |
| Large model inference (GPT-2, batch=32) | 1.2–2x |
| Training (full step) | 1.1–1.5x |

## Basic API Walkthrough

Three steps: **warmup** → **capture** → **replay**

In [ ]:
if HAS_CUDA:
    # A simple model
    model = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 10),
    ).cuda().eval()

    # Pre-allocate static input
    static_input = torch.randn(64, 512, device="cuda")

    # Step 1: Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = model(static_input)
    torch.cuda.synchronize()

    # Step 2: Capture
    g = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(g):
            static_output = model(static_input)

    print(f"Graph captured!")
    print(f"Input shape:  {static_input.shape}")
    print(f"Output shape: {static_output.shape}")

    # Step 3: Replay with new data
    new_data = torch.randn(64, 512, device="cuda")
    static_input.copy_(new_data)
    g.replay()
    torch.cuda.synchronize()

    # Verify correctness
    with torch.no_grad():
        eager_out = model(new_data)
    print(f"Max diff (graph vs eager): {(static_output - eager_out).abs().max().item():.2e}")
else:
    print("No CUDA GPU — see README for code walkthrough")

## Static Inputs Explained

CUDA Graphs record **memory addresses**, not values. During replay, the GPU reads from the exact same addresses that were captured.

This means:
1. **Pre-allocate** input tensors before capture
2. **`copy_()`** new data into them before each replay
3. **Read** results from the pre-allocated output tensor

```python
# CORRECT:
static_input.copy_(new_batch)   # Same address, new values
g.replay()

# WRONG:
new_input = data.cuda()          # New address each time!
g.replay()                       # Graph reads from OLD address
```

In [ ]:
if HAS_CUDA:
    static_in = torch.zeros(32, 256, device="cuda")
    sm = nn.Linear(256, 64).cuda().eval()

    with torch.no_grad():
        for _ in range(3):
            _ = sm(static_in)

    g2 = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(g2):
            static_out = sm(static_in)

    print("Processing batches through the same graph:")
    for i in range(5):
        batch = torch.randn(32, 256, device="cuda")
        static_in.copy_(batch)
        g2.replay()
        torch.cuda.synchronize()
        print(f"  Batch {i}: output norm = {static_out.norm().item():.4f}")

    del g2, sm
else:
    print("No CUDA GPU")

## Warmup Pattern

Before capturing a CUDA Graph, you **must** run the model at least once. Warmup triggers lazy initialization:

| Lazy Initialization | Why |
|---------------------|-----|
| cuDNN algorithm selection | First conv benchmarks algorithms |
| CUDA context creation | First CUDA call inits the driver |
| Memory allocator warmup | Caching allocator builds its pool |
| cuBLAS handle creation | First matmul creates the handle |

Without warmup, capture records these one-time operations — causing bloated graphs or `RuntimeError` from dynamic allocations.

In [ ]:
if HAS_CUDA:
    conv_model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10),
    ).cuda().eval()

    static_img = torch.randn(8, 3, 32, 32, device="cuda")

    # Warmup
    print("Running warmup iterations:")
    with torch.no_grad():
        for i in range(3):
            out = conv_model(static_img)
            torch.cuda.synchronize()
            print(f"  Warmup {i+1}: output shape {out.shape}")

    # Now safe to capture
    g3 = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(g3):
            conv_out = conv_model(static_img)

    print(f"Conv model graph captured. Output shape: {conv_out.shape}")
    del g3, conv_model
else:
    print("No CUDA GPU")

## Benchmark: Eager vs CUDA Graph Replay

Let's measure the actual speedup on a small MLP where CPU overhead dominates.

In [ ]:
if HAS_CUDA:

    class SmallMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(256, 128), nn.ReLU(),
                nn.Linear(128, 64), nn.ReLU(),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, 10),
            )
        def forward(self, x):
            return self.layers(x)

    bench_model = SmallMLP().cuda().eval()
    bench_input = torch.randn(16, 256, device="cuda")
    N = 1000

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = bench_model(bench_input)
    torch.cuda.synchronize()

    # Eager benchmark
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N):
            _ = bench_model(bench_input)
    torch.cuda.synchronize()
    eager_ms = (time.perf_counter() - t0) * 1000

    # Capture graph
    g_bench = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(g_bench):
            bench_out = bench_model(bench_input)

    # Graph replay benchmark
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N):
        g_bench.replay()
    torch.cuda.synchronize()
    graph_ms = (time.perf_counter() - t0) * 1000

    print(f"Small MLP benchmark ({N} iterations):")
    print(f"  Eager:      {eager_ms:.1f} ms  ({eager_ms/N*1000:.1f} us/iter)")
    print(f"  CUDA Graph: {graph_ms:.1f} ms  ({graph_ms/N*1000:.1f} us/iter)")
    print(f"  Speedup:    {eager_ms/graph_ms:.2f}x")

    del g_bench, bench_model
else:
    print("No CUDA GPU — typical speedup: 3-8x for small MLPs")

## torch.compile with reduce-overhead Mode

The **easiest** way to use CUDA Graphs is through `torch.compile`:

```python
compiled = torch.compile(model, mode="reduce-overhead")
output = compiled(input_tensor)  # Graphs managed automatically
```

Under the hood, `reduce-overhead` uses Inductor's **cudagraph_trees**:
- Automatic warmup and capture
- Per-region graphs (partial capture OK)
- Kernel fusion + graph replay combined
- Shape-variant caching

| Manual CUDAGraph | torch.compile reduce-overhead |
|-----------------|-------------------------------|
| You manage static inputs | Automatic |
| You do warmup | Automatic |
| Entire model must be graphable | Per-region (partial) |
| No kernel fusion | Fusion + graphs combined |
| Fixed shapes only | Multiple shapes cached |

In [ ]:
if HAS_CUDA:

    class DemoModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(512, 256)
            self.fc2 = nn.Linear(256, 128)
            self.fc3 = nn.Linear(128, 10)

        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            return self.fc3(x)

    demo_model = DemoModel().cuda().eval()
    demo_input = torch.randn(32, 512, device="cuda")

    compiled_model = torch.compile(demo_model, mode="reduce-overhead")

    # First call triggers compilation + capture
    with torch.no_grad():
        out = compiled_model(demo_input)
    torch.cuda.synchronize()
    print(f"Compiled output shape: {out.shape}")

    # Benchmark
    with torch.no_grad():
        for _ in range(10):
            _ = compiled_model(demo_input)
    torch.cuda.synchronize()

    N = 500
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N):
            _ = compiled_model(demo_input)
    torch.cuda.synchronize()
    compiled_ms = (time.perf_counter() - t0) * 1000
    print(f"reduce-overhead: {compiled_ms:.1f} ms for {N} iters ({compiled_ms/N*1000:.1f} us/iter)")

    del compiled_model, demo_model
else:
    print("No CUDA GPU")

## Limitations — What Breaks Inside CUDA Graphs

### Hard Failures (RuntimeError)

| Operation | Why |
|-----------|-----|
| `print(tensor)` | CPU sync |
| `tensor.item()` | Transfers to CPU |
| `tensor.cpu()` | Cross-device copy |
| `torch.tensor([1,2,3])` | CPU tensor creation |
| Dynamic memory allocation | Variable-size allocs |

### Silent Failures (Wrong Results)

| Pattern | Problem |
|---------|---------|
| `if x > 0:` (data-dependent) | Branch frozen at capture |
| Dynamic shapes | Dimensions hardcoded |
| Random ops without seed | Same values every replay |

In [ ]:
if HAS_CUDA:
    print("Demo: data-dependent control flow is FROZEN at capture time\n")

    class ConditionalModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear = nn.Linear(64, 64)

        def forward(self, x):
            out = self.linear(x)
            if out.sum() > 0:
                return out * 2
            else:
                return out * 0.5

    cond_model = ConditionalModel().cuda().eval()
    cond_input = torch.randn(4, 64, device="cuda")

    with torch.no_grad():
        for _ in range(3):
            _ = cond_model(cond_input)

    g_cond = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(g_cond):
            cond_out = cond_model(cond_input)

    # Change input drastically
    cond_input.copy_(torch.randn(4, 64, device="cuda") * 100)
    g_cond.replay()
    torch.cuda.synchronize()

    with torch.no_grad():
        eager_out = cond_model(cond_input)

    diff = (cond_out - eager_out).abs().max().item()
    print(f"Graph vs eager max diff: {diff:.4f}")
    if diff > 0.01:
        print("-> Branch was FROZEN at capture time (different branch taken in eager)")
    else:
        print("-> Same branch happened to be taken (try running again)")

    del g_cond, cond_model
else:
    print("No CUDA GPU")

## CUDA Graphs with AMP

Autocast works inside capture — the graph records mixed-precision kernel variants. At replay time, no autocast context is needed.

In [ ]:
if HAS_CUDA:
    amp_model = nn.Sequential(
        nn.Linear(512, 256), nn.ReLU(),
        nn.Linear(256, 10),
    ).cuda().eval()

    amp_input = torch.randn(32, 512, device="cuda")

    # Warmup with AMP
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(3):
            _ = amp_model(amp_input)
    torch.cuda.synchronize()

    # Capture with AMP
    g_amp = torch.cuda.CUDAGraph()
    with torch.no_grad(), torch.amp.autocast("cuda"):
        with torch.cuda.graph(g_amp):
            amp_out = amp_model(amp_input)

    print(f"AMP graph output dtype: {amp_out.dtype}")

    # Replay (no autocast needed)
    amp_input.copy_(torch.randn(32, 512, device="cuda"))
    g_amp.replay()
    torch.cuda.synchronize()
    print(f"Replay output norm: {amp_out.norm().item():.4f}")

    del g_amp, amp_model
else:
    print("No CUDA GPU")

## make_graphed_callables

`torch.cuda.make_graphed_callables` wraps a module with automatic warmup, capture, and static-input management:

```python
graphed = torch.cuda.make_graphed_callables(
    model, sample_args=(sample_input,), num_warmup_iters=3
)
output = graphed(input_tensor)
```

In [ ]:
if HAS_CUDA:
    mgc_model = nn.Sequential(
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 10),
    ).cuda().eval()

    sample = torch.randn(16, 128, device="cuda")

    with torch.no_grad():
        graphed = torch.cuda.make_graphed_callables(
            mgc_model,
            sample_args=(sample,),
            num_warmup_iters=3,
        )

    with torch.no_grad():
        result = graphed(sample)
        eager_result = mgc_model(sample)

    print(f"Output shape: {result.shape}")
    print(f"Max diff vs eager: {(result - eager_result).abs().max().item():.2e}")
    del graphed, mgc_model
else:
    print("No CUDA GPU")

## Graph Pool Sharing

Multiple graphs can share a memory pool to reduce total GPU memory usage. Use when graphs execute sequentially (never concurrently).

```python
with torch.cuda.graph(g1):
    out1 = model(input1)

with torch.cuda.graph(g2, pool=g1.pool()):
    out2 = model(input2)
```

In [ ]:
if HAS_CUDA:
    pool_model = nn.Linear(256, 64).cuda().eval()
    in_a = torch.randn(8, 256, device="cuda")
    in_b = torch.randn(16, 256, device="cuda")

    with torch.no_grad():
        for _ in range(3):
            _ = pool_model(in_a)
            _ = pool_model(in_b)

    ga = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(ga):
            out_a = pool_model(in_a)

    gb = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(gb, pool=ga.pool()):
            out_b = pool_model(in_b)

    ga.replay()
    torch.cuda.synchronize()
    print(f"Graph A output: {out_a.shape}")

    gb.replay()
    torch.cuda.synchronize()
    print(f"Graph B output: {out_b.shape}")
    print("Both graphs share a memory pool (sequential use only)")

    del ga, gb, pool_model
else:
    print("No CUDA GPU")

## Practical Pattern: Inference Server

The most common CUDA Graph use case — serving predictions at high throughput with fixed batch size.

In [ ]:
if HAS_CUDA:

    class GraphedInferenceServer:
        def __init__(self, model, batch_size, input_dim):
            self.model = model.cuda().eval()
            self.static_input = torch.zeros(batch_size, input_dim, device="cuda")
            self.graph = torch.cuda.CUDAGraph()

            with torch.no_grad():
                for _ in range(3):
                    _ = self.model(self.static_input)
            torch.cuda.synchronize()

            with torch.no_grad():
                with torch.cuda.graph(self.graph):
                    self.static_output = self.model(self.static_input)

        def predict(self, input_tensor):
            self.static_input.copy_(input_tensor)
            self.graph.replay()
            return self.static_output.clone()

    server_model = nn.Sequential(
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 10),
    )
    server = GraphedInferenceServer(server_model, batch_size=8, input_dim=128)

    print("Serving 10 requests:")
    for i in range(10):
        request = torch.randn(8, 128, device="cuda")
        pred = server.predict(request)
        if i < 3 or i == 9:
            print(f"  Request {i}: predictions = {pred.argmax(dim=1).tolist()}")
        elif i == 3:
            print("  ...")

    del server
else:
    print("No CUDA GPU")

## Partial Graph Capture for Training

Full training steps are hard to graph (optimizer state, gradient scaling). Instead, capture only the forward+backward pass and run the optimizer eagerly.

In [ ]:
if HAS_CUDA:
    train_model = nn.Sequential(
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),
    ).cuda()
    optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)

    static_x = torch.randn(32, 256, device="cuda")
    static_y = torch.randn(32, 10, device="cuda")

    # Warmup
    for _ in range(3):
        out = train_model(static_x)
        loss = nn.functional.mse_loss(out, static_y)
        loss.backward()
        optimizer.zero_grad()

    # Capture forward + backward
    g_train = torch.cuda.CUDAGraph()
    with torch.cuda.graph(g_train):
        out = train_model(static_x)
        loss = nn.functional.mse_loss(out, static_y)
        loss.backward()

    # Training loop
    print("Training with partial graph capture:")
    for step in range(20):
        static_x.copy_(torch.randn(32, 256, device="cuda"))
        static_y.copy_(torch.randn(32, 10, device="cuda"))
        g_train.replay()
        optimizer.step()
        optimizer.zero_grad()
        if step % 5 == 0:
            torch.cuda.synchronize()
            print(f"  Step {step}: loss = {loss.item():.4f}")

    del g_train, train_model
else:
    print("No CUDA GPU")

## Decision Tree: When to Use CUDA Graphs

```
Is your model on NVIDIA GPU?
+-- No  -> Not applicable
+-- Yes
    |
    Fixed input shapes?
    +-- No  -> torch.compile (handles dynamic shapes)
    +-- Yes
        |
        CPU launch overhead is bottleneck?
        (small model, many kernels, high throughput)
        +-- No  -> torch.compile for kernel fusion
        +-- Yes
            |
            Inference only?
            +-- Yes -> CUDA Graphs ideal!
            |          reduce-overhead or manual API
            +-- No  -> Partial capture (fwd+bwd)
                       or reduce-overhead
```

| Scenario | Recommendation |
|----------|---------------|
| Inference, fixed shapes | Manual CUDAGraph or `reduce-overhead` |
| Inference, variable shapes | `torch.compile(mode="default")` |
| Training, single GPU | `reduce-overhead` |
| Training, multi-GPU | `torch.compile(mode="default")` (NCCL compat) |
| Data-dependent control flow | `torch.compile` with graph breaks |
| Quick experiment | `make_graphed_callables` |

## Exercise: Capture and Benchmark a 3-Layer MLP

**Task:** Build a 3-layer MLP (512 → 256 → 128 → 10), capture it with CUDA Graphs, and compare eager vs graph replay throughput.

Steps:
1. Define the model and move to GPU
2. Create a static input (batch=32, features=512)
3. Warmup (3 iterations)
4. Capture the graph
5. Benchmark both eager and graph replay over 1000 iterations
6. Print the speedup

In [ ]:
if HAS_CUDA:
    # Your code here!
    # Hint: Follow the capture/replay pattern from earlier cells.

    # 1. Define model
    exercise_model = nn.Sequential(
        nn.Linear(512, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),
    ).cuda().eval()

    # 2. Static input
    ex_input = torch.randn(32, 512, device="cuda")

    # 3. Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = exercise_model(ex_input)
    torch.cuda.synchronize()

    # 4. Capture
    ex_graph = torch.cuda.CUDAGraph()
    with torch.no_grad():
        with torch.cuda.graph(ex_graph):
            ex_output = exercise_model(ex_input)

    # 5. Benchmark
    N = 1000

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N):
            _ = exercise_model(ex_input)
    torch.cuda.synchronize()
    eager_ms = (time.perf_counter() - t0) * 1000

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N):
        ex_graph.replay()
    torch.cuda.synchronize()
    graph_ms = (time.perf_counter() - t0) * 1000

    # 6. Results
    print(f"3-Layer MLP Exercise Results ({N} iterations):")
    print(f"  Eager:      {eager_ms:.1f} ms")
    print(f"  CUDA Graph: {graph_ms:.1f} ms")
    print(f"  Speedup:    {eager_ms/graph_ms:.2f}x")

    del ex_graph, exercise_model
else:
    print("This exercise requires a CUDA GPU.")
    print("Try running in Google Colab with a free GPU runtime.")

## Key Takeaways

1. **CUDA Graphs** capture GPU operations into a replayable graph, eliminating per-kernel CPU launch overhead

2. **Static inputs**: Pre-allocate tensors, `copy_()` new data in, `replay()`, read from pre-allocated output

3. **Always warmup** before capture (3+ forward passes) to trigger lazy initialization

4. **`torch.compile(mode="reduce-overhead")`** is the easiest path — handles warmup, capture, pools, and partial graphs automatically

5. **Limitations**: No dynamic shapes, no CPU sync (`item()`, `cpu()`, `print()`), no data-dependent control flow, no dynamic memory allocation

6. **Best for**: Inference servers with fixed batch sizes and small/medium models where CPU overhead dominates

7. **Pool sharing** reduces memory when multiple graphs execute sequentially

8. For training or dynamic shapes, prefer `torch.compile(mode="default")` instead

---

**Next steps:** Review Module 08 (torch.compile) for how CUDA Graphs integrate with the compilation pipeline, or explore Module 20 (Backends Tuning) for other performance knobs.

In [ ]:
if HAS_CUDA:
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e6:.1f} MB")
print("\nNotebook complete!")